In [ ]:
import itertools
import logging
import math
import os
from glob import glob

import numpy as np
import pandas
import rasterio
from tqdm import tqdm

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s", level=logging.INFO)

In [ ]:
os.chdir(
    "/soge-home/projects/mistral/jamaica-ccri/processed_data/hazards/jba-flood-events/"
)

In [ ]:
def read_events(event_set, hazard_code):
    """Read calculated event values

    Parameters
    ----------
    event_set : str
        Short name for event set (ObsEventRP, SimEventRP), corresponds
        to directory.
    hazard_code : str
        FLRF or FLSW, corresponds to river or surface flooding and event
        data file naming pattern.
    """
    # Read all events
    fnames = sorted(glob(f"{event_set}/*/*__{hazard_code}__*"))
    event_dfs = []
    for fname in tqdm(fnames):
        try:
            event_df = read_event(fname)
            if not event_df.empty:
                event_dfs.append(event_df)
        except IndexError:
            logging.warning(f"Skipping {fname} with IndexError")
    return event_dfs


def filter_events(event_dfs, imin, imax):
    filtered_dfs = [df.loc[imin : (imax - 1)] for df in event_dfs]
    events = pandas.concat(filtered_dfs, axis=1).fillna(0)
    return events


def read_event(fname):
    """Read a single event data file"""
    df = pandas.read_parquet(fname)
    event = df.iloc[0, 2]
    return (
        df[["depth", "cell_index"]]
        .set_index("cell_index")
        .rename(columns={"depth": event})
        .sort_index()
        .copy()
    )


def calculate_rps(events, rps, nyears):
    """Calculate return-period depth values

    Parameters
    ----------
    events : pandas.DataFrame
        Event depth data, index should be cell_index of exposure points,
        columns should be one per event, so a row contains depth values
        for all events at one exposure point.
    rps : np.array
        List of return periods to calculate.
    nyears : int
        Number of years represented by event set.
    """
    # DataFrame to 2D numpy array
    depths = events.to_numpy()

    # pad to nyears of events
    ncols = len(events.columns)
    depths = np.pad(depths, ((0, 0), (0, nyears - ncols)))

    # calculate return-period values from quantiles
    quantiles = 1 - (1 / rps)
    depth_quantiles = np.quantile(depths, quantiles, axis=1)

    # set up return-period dataframe
    events["rp0002"] = 0
    ep_rps = events[["rp0002"]].copy()

    for i, rp in enumerate(rps):
        col = f"rp{str(rp).zfill(4)}"
        ep_rps[col] = depth_quantiles[i]

    return ep_rps

In [ ]:
# with rasterio.open('inputs/fluvial_raw_fld_depth/JM_FLRF_UD_Q20_RD_02-aligned.tif') as ds:
#     pass

# ds.width, ds.height, ds.width * ds.height

In [ ]:
# fnames = sorted(glob(f"ObsEventRP/*/*__FLRF__*"))
# fnames[0]

In [ ]:
# df = read_event(fnames[0])
# df

In [ ]:
event_sets = sorted(glob("SimEventRP*0s") + ["SimEventRP"])
event_sets

In [ ]:
# obs_events = read_events('ObsEventRP', 'FLRF')

In [ ]:
# filter_events(obs_events, 0, int(1e9))

In [ ]:
rps = np.array([2, 20, 50, 100, 200, 500, 1500])
# ep_rps = calculate_rps(obs_events, rps, 30)

In [ ]:
# ep_rps.to_parquet('ObsEventRP_FLRF.parquet')

In [ ]:
def process_step(event_set, hazard_code, rps, nblocks, stride):
    event_dfs = read_events(event_set, hazard_code)
    for block in tqdm(range(nblocks)):
        cell_index_min = block * stride
        cell_index_max = cell_index_min + stride
        events = filter_events(event_dfs, cell_index_min, cell_index_max)
        ep_rps = calculate_rps(events, rps, int(1e4))
        ep_rps.to_parquet(f"scratch/eprp_{event_set}_{hazard_code}_{block}.parquet")

In [ ]:
flrf_eps = pandas.read_parquet("flrf_exposure_points.gpq")
flsw_eps = pandas.read_parquet("flsw_exposure_points.gpq")

In [ ]:
stride = 1e6
nblocks = math.ceil(max(flrf_eps.cell_index.max(), flsw_eps.cell_index.max()) / stride)
nblocks

In [ ]:
for event_set, hazard_code in itertools.product(event_sets, ("FLRF", "FLSW")):
    logging.info(f"{hazard_code}, {event_set}")
    process_step(event_set, hazard_code, rps, nblocks, stride)